# 5-1 Assignment: Cartpole Problem
___
<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Run all code blocks and observe the output. Once you have reviewed the sample output. Use the <b>LastName_FirstName_Assignment2.ipynb</b> file to complete your assignment.
</div>

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> For compatability purposes, libraries have been updated from those used in the required readings to match to current versions; hence some of the package invocations may differ slightly from the book. The affected lines of code have comments added to the right as applicable, with the old code commented out above for reference.
</div>

<div class="alert alert-block alert-danger" style="color:black;">
<b>GPU/CUDA/Memory Warnings/Errors:</b> You may receive some errors referencing that GPUs will not be used, CUDA could not be found, or free system memory allocation errors. These and a few others, are standard errors that can be ignored here as they are environment based.<br><br>
<b>Example messages:</b>
    <ul>
        <li>Could not find cuda drivers on your machine, GPU will not be used.</li>
        <li>Please check linkage and avoid linking the same target more than once.</li>
        <li>E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)</li>
        <li>Allocation of ######## exceeds 10% of free system memory</li>
    </ul>
</div>

---

### Installing Required Packages
This is to install necessary components to run the assignment

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> If the assignment takes too long, you may want to download the code and run it locally to take advantage of your GPU.
</div>

In [2]:
# Package Imports
import numpy as np
import tensorflow as tf
import gymnasium as gym
from collections import deque
import random

# Environment setup and Variables
try:
    env = gym.make("CartPole-v1", render_mode=None)
except Exception:
    print('Failed to initialize environment! Make sure that gymnasium was installed correctly!')
    sys.exit(1)

state_shape = env.observation_space.shape[0]
action_size = env.action_space.n

# Hyperparameters
GAMMA = 0.75
EXPLORATION_MAX = 0.05
EXPLORATION_MIN = 0.001
EXPLORATION_DECAY = 0.01
LEARNING_RATE = 0.0051
BATCH_SIZE = 64
TRAIN_START = 1000
MEMORY_SIZE = 2000
EPISODES = 300

# Counters during training
train_freq = 4
step_count = 0
target_update_freq = 10
# If an error occurs below, it is because the environment is looking for a GPU.

In [3]:
memory = deque(maxlen=MEMORY_SIZE)

# DQN Builder
def build_dqn_model():
    try:
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(state_shape,)),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(action_size, activation='linear')
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
        return model
    except Exception:
        print('Error occurred while building the DQN model!')
        sys.exit(1)

In [4]:
# Calling the build_dqn_model method
model = build_dqn_model()
target_model = build_dqn_model()
target_model.set_weights(model.get_weights())

I0000 00:00:1791764643.614434   94950 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791764643.687802   94950 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791764643.690645   94950 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791764643.697649   94950 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> If an error displays, it is simply trying to connect to the GPU.
</div>


In [5]:
# Random Action selection for exploration
def get_action(state, EXPLORATION_MAX):
    try:
        if np.random.rand() <= EXPLORATION_MAX:
            return random.randrange(action_size)
        q_values = model.predict(np.array([state]), verbose=0) #Remove verbose=0 to see every prediction display
        return np.argmax(q_values[0])
    except Exception:
            print("Error during get_action method!")
            sys.exit(1)

In [6]:
# Experience Replay
def experience_replay():
    if len(memory) < TRAIN_START:
        return

    batch = random.sample(memory, BATCH_SIZE)
    states = np.zeros((BATCH_SIZE, state_shape))
    next_states = np.zeros((BATCH_SIZE, state_shape))
    actions, rewards, completions = [], [], []

    try:
        for i, (state, action, reward, state_next, terminal) in enumerate(batch):
            states[i] = state
            next_states[i] = state_next
            actions.append(action)
            rewards.append(reward)
            completions.append(terminal)

        target_q = model.predict(states, verbose=0) #Remove verbose=0 to see every prediction display
        next_q = target_model.predict(next_states, verbose=0) #Remove verbose=0 to see every prediction display

        for i in range(BATCH_SIZE):
            if completions[i]:
                target_q[i][actions[i]] = rewards[i]
            else:
                target_q[i][actions[i]] = rewards[i] + GAMMA * np.max(next_q[i])

        model.fit(states, target_q, batch_size=BATCH_SIZE, verbose=0)
    except Exception:
        print("Error during replay")
        sys.exit(1)

In [ ]:
# A list to keep track of all of the episodes
episode_rewards = []

# Main loop
# Will iterate through based on the number of EPISODES originally listed up top
for e in range(EPISODES):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = get_action(state, EXPLORATION_MAX)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        reward = reward if not done else -10

        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward
        
        step_count += 1
        if step_count % train_freq == 0:
            experience_replay()

    if EXPLORATION_MAX > EXPLORATION_MIN: 
        EXPLORATION_MAX *= EXPLORATION_DECAY
        EXPLORATION_MAX = max(EXPLORATION_MIN, EXPLORATION_MAX)

    if e % target_update_freq == 0:
        target_model.set_weights(model.get_weights())

    episode_rewards.append(total_reward)
    # This will display the average reward of every 10 episodes occurring.
    # Comment this section and add the below print method if you want to display every episode
    if e % 10 == 0:
        avg_reward = np.mean(episode_rewards[-10:])
        print(f"Episode {e+1}: Average Reward = {avg_reward:.2f}, Exploration = {EXPLORATION_MAX:.3f}")
    
    # Logic to complete execution if average is greater than 195
    if len(episode_rewards) >= 100:
        avg_last_hun = np.mean(episode_rewards[-100:])
        if avg_last_hun >= 195:
            print(f"Solved at episode {e}: Average reward over last 100: {avg_last_hun:.2f}")
            break
        
    # Comment this in if you want to see every iteration of the training occurring     
    # print(f"Episode {e+1}: Total Reward = {total_reward}, Exploration = {EXPLORATION_MAX:.3f}")

# Saving model as needed    
model.save("Cartpole_model.h5")
print("Training complete! Model saved as Cartpole_model")

I0000 00:00:1791764688.121737   95018 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Episode 1: Average Reward = -1.00, Exploration = 0.001
Episode 11: Average Reward = -1.30, Exploration = 0.001
Episode 21: Average Reward = -1.20, Exploration = 0.001
Episode 31: Average Reward = -1.40, Exploration = 0.001
Episode 41: Average Reward = -1.70, Exploration = 0.001
Episode 51: Average Reward = -1.60, Exploration = 0.001
Episode 61: Average Reward = -1.70, Exploration = 0.001
Episode 71: Average Reward = -1.80, Exploration = 0.001
Episode 81: Average Reward = -1.80, Exploration = 0.001
Episode 91: Average Reward = -1.30, Exploration = 0.001
Episode 101: Average Reward = -1.50, Exploration = 0.001
Episode 111: Average Reward = 32.00, Exploration = 0.001
Episode 121: Average Reward = 3.40, Exploration = 0.001
Episode 131: Average Reward = -1.40, Exploration = 0.001
Episode 141: Average Reward = -1.30, Exploration = 0.001
Episode 151: Average Reward = -1.50, Exploration = 0.001
Episode 161: Average Reward = -0.90, Exploration = 0.001
Episode 171: Average Reward = -1.00, Explor

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b><br>
    If an error displays, it is attempting to connect to the GPU.<br>
    It will not connect and run on CPU. <br>
    Code will take some time to run, which is commonplace for real life models.<br>
    <b><i>-- Make sure your computer does not go to sleep, and take a well-deserved break! --</b></i>
</div>


<div class="alert alert-block alert-success" style="color:black;">
<b>Make your observations here as it pertains to the assignment rubric.</b> 
</div>


The goal of reinforcement learning in the Cartpole problem is to train an agent to balance an inverted pendulum by learning through trial and error and maximizing its rewards. The agent is represented by the DQNSolver, which uses the Deep Q-Learning (DQN) algorithm to determine the best action based on the current state of the environment. The four state values are cart position, cart velocity, pole angle, and pole angular velocity. The two possible actions are moving the cart left or right to keep the pole balanced. Experience replay allows the algorithm to store previous experiences in memory and randomly sample batches of them to train the neural network. This helps reduce the correlation between consecutive experiences and allows the agent to learn from a wider range of situations (Surma, 2021). The discount factor, GAMMA, determines how much the agent values future rewards compared to immediate rewards. I changed GAMMA from 0.99 to 0.75 to see how reducing the importance of future rewards affected the algorithm. A value closer to 1 encourages the agent to consider long-term rewards, while a lower value places more emphasis on immediate rewards. A neural network is used instead of a traditional Q-table because it can estimate Q-values for the agent's possible actions based on the four state variables. This approach is more practical when working with environments that have many possible states, and the network uses the Bellman equation to update its estimates of future rewards (Kurban, 2022; Surma, 2021).

During my experiments, I changed the learning rate and exploration settings to observe their effects on the agent's performance. I tested learning rates of 0.001, 0.0001, and 0.0051. The learning rate determines the size of the updates made to the neural network during training. A lower learning rate makes these updates smaller and more gradual, while a higher learning rate can cause larger changes that may make learning less stable. In my earlier observations, the default learning rate of 0.001 completed training in approximately 125 episodes, the 0.0001 setting required approximately 192 episodes, and the 0.01 setting took more than 784 episodes before the simulation was terminated. These earlier results suggest that the learning rate can have a significant effect on training time, although the runs were not all completed successfully under the later experiments. I also changed the exploration settings to EXPLORATIONMAX = 0.05, EXPLORATIONMIN = 0.001, and EXPLORATION_DECAY = 0.01. With these settings, the reported average reward increased from -3.00 in episode 1 to 30.00 in episode 171, indicating that the agent was beginning to improve. However, the exploration rate dropped to its minimum very quickly, which limited the agent's opportunity to try different actions. Several earlier experiments also encountered TensorFlow GPU initialization errors, so I could not make a reliable performance comparison across every parameter change. 
